# LLM from scratch · the series notebook (Transformers from scratch)

One notebook for the whole YouTube series: GPT-2 small, opened up step by step, until we train our own GPT-2-style models and the
modern variants. Each chapter builds on the one before. This notebook goes up to **Chapter 2** and matches Episode 2. Each chapter's notebook has all the chapters before it, so the newest one has everything: [all chapters](https://github.com/gotherohit/llm-scratch#chapters).

**How it works**
- Run **Setup** first, every time you open the notebook.
- Each chapter needs only Setup plus the **toolkit** cells of earlier chapters (the last cell of each chapter), so you can jump
  straight to any chapter.
- Cells marked **✏️ YOUR TURN** have blanks (`___`). Fill them in and run; the check cell below prints ✅ when you're right.
  "Run all" stops at the first blank; that's expected. Solutions are at the end of each chapter.

**Chapters**
1. Chapter 1 · Text → numbers → vectors
2. Chapter 2 · Attention by hand

## Setup
Loads GPT-2 small (about 500 MB, downloaded once per session). From chapter 1 on we also prepare the **TinyStories** dataset that the
training chapters will train on. To keep those files (and later the checkpoints) when the session ends, set `USE_DRIVE = True`:
Colab will ask you to sign in to Google and allow access to your Drive. Everything goes into one folder, `MyDrive/transformer-scratch-yt/`.

In [ ]:
USE_DRIVE = False          # False: files are temporary (fine for chapter 1). Training chapters will ask you to switch this on.

import os
import numpy as np
import torch
from transformers import GPT2TokenizerFast, GPT2LMHeadModel

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")          # opens Google's sign-in and permission prompt
    WORK = "/content/drive/MyDrive/transformer-scratch-yt"
else:
    WORK = "/content/transformer-scratch-yt"   # temporary: deleted when the session ends
for sub in ["data", "checkpoints", "runs"]:
    os.makedirs(os.path.join(WORK, sub), exist_ok=True)

tok = GPT2TokenizerFast.from_pretrained("openai-community/gpt2")
model = GPT2LMHeadModel.from_pretrained("openai-community/gpt2").eval()
print("GPT-2 small loaded:", f"{sum(p.numel() for p in model.parameters()):,}", "parameters · files go to", WORK)

---
## Chapter 1 · Text → numbers → vectors
You'll turn **"The cat sat on"** into the exact table of numbers GPT-2 sends into its first layer, using plain numpy:
token IDs, the embedding table, positions, batches, and how the lookup works as math. Then you do the same on the real dataset we'll
train on later, **TinyStories**: tokenize it, save it as `train.bin` / `val.bin`, and cut real training batches. Last, you start
**TinyGPT**, our own model in PyTorch, which grows one piece per chapter until we train it. Needs only **Setup**.

### 1.0 · Numpy warm-up
🎬 **Video walkthrough:** [Deep dive D1.1 · 3-D Matrix Multiplication, Finally Visual](https://youtu.be/25GaBn2QYW4)

A numpy **array** is a table of numbers. `.shape` tells you its size as `(rows, columns)`.

In [ ]:
import numpy as np

A = np.array([[1, 2, 0],
              [0, 1, 3]])
print(A.shape)      # (2, 3): 2 rows, 3 columns
print(A[1])         # row 1 (counting starts at 0): [0 1 3]
print(A[[1, 0]])    # rows 1 and 0, in that order
print(A + A)        # adding works number by number

**Matrix multiplication** uses `@`. Shape rule: `[a, b] @ [b, c] → [a, c]`.

In [ ]:
B = np.array([[2, 1],
              [0, 1],
              [1, 4]])
C = A @ B
print(C.shape)   # (2, 2)
print(C)         # top-left = 1*2 + 2*0 + 0*1 = 2

✏️ **YOUR TURN:** compute the top-left number of `C` by hand with a dot product: row 0 of `A` times column 0 of `B`, pair by pair, then added up. (Hint: `B[:, 0]` is column 0, and `np.sum(...)` adds things up.)

In [ ]:
top_left = ___

In [ ]:
assert top_left == C[0, 0]; print("✅ that's exactly what @ does for every row and column")

### 1.1 · Look inside GPT-2 small
Setup already loaded the real model as `model`. We only take two tables out of it.

In [ ]:
wte = model.transformer.wte.weight.detach().numpy()   # token embedding table
wpe = model.transformer.wpe.weight.detach().numpy()   # position table
print("wte", wte.shape, " wpe", wpe.shape)

Before running the next cell, predict: how many numbers are in `wte`? (rows × columns)

In [ ]:
print(f"{wte.size:,} numbers in the token table")
print(f"{sum(p.numel() for p in model.parameters()):,} numbers in the whole model")

### 1.2 · Text → token IDs
GPT-2's code always works on a **batch**: a table of IDs with shape `[B, T]` (B texts, T tokens each). One sentence is a batch of 1,
so our IDs have shape `[1, 4]`.

✏️ **YOUR TURN:** use the tokenizer to get the IDs as a numpy table. (Hint: `tok(text, return_tensors="np")["input_ids"]`.)

In [ ]:
text = "The cat sat on"
ids = ___
print(ids.shape, ids, [tok.decode([i]) for i in ids[0]])

In [ ]:
assert ids.shape == (1, 4) and ids.tolist() == [[464, 3797, 3332, 319]]
print("✅ [1, 4]: B = 1 sentence, T = 4 tokens. Note the space at the front of ' cat'")

### 1.3 · IDs → rows of the table
The lookup keeps the IDs' shape and adds one row's 768 numbers on the end: `[1, 4]` → `[1, 4, 768]`.

✏️ **YOUR TURN:** pick the rows of `wte` for our IDs. (Indexing with the whole `[1, 4]` table works in one go.)

In [ ]:
tok_rows = ___
print(tok_rows.shape)
print(tok_rows[0, :, :4].round(4))   # first 4 numbers of each token's row

In [ ]:
assert tok_rows.shape == (1, 4, 768) and np.allclose(tok_rows[0, 1], wte[3797])
print("✅ one row per token: [1, 4] → [1, 4, 768]")

### 1.4 · The same lookup, as a matrix multiplication
🎬 **Video walkthrough:** [Deep dive D1.1 · 3-D Matrix Multiplication, Finally Visual](https://youtu.be/25GaBn2QYW4)

A **one-hot** row is all zeros except a 1 at the ID's position. One-hot `[1, 4, 50257]` @ table `[50257, 768]` → `[1, 4, 768]`.
(The extra 1 in front just comes along; section 1.11 shows exactly how multiplication works with 3 dimensions.)

✏️ **YOUR TURN:** fill in the 1s, then multiply.

In [ ]:
onehot = np.zeros((1, 4, 50257), dtype=np.float32)
for t, i in enumerate(ids[0]):
    onehot[0, t, ___] = 1.0

via_matmul = ___
print(via_matmul.shape)

In [ ]:
assert np.array_equal(via_matmul, tok_rows); print("✅ identical: a lookup is a multiplication by one-hot rows")

### 1.5 · Add the position rows
Positions don't depend on which sentence it is, so the position rows have no batch dimension: rows 0 to 3 of `wpe`, shape `[4, 768]`.
Adding `[4, 768]` to `[1, 4, 768]` adds them to every sentence in the batch (here just one). That's called **broadcasting**.

✏️ **YOUR TURN:** take rows 0 to 3 of `wpe` and add them to `tok_rows`.

In [ ]:
pos_rows = ___
x = ___
print(pos_rows.shape, x.shape)

Now the real test: ask GPT-2 what it sends into layer 1 (`hidden_states[0]`) and compare.

In [ ]:
with torch.no_grad():
    out = model(torch.tensor(ids), output_hidden_states=True)
gpt2_x = out.hidden_states[0].numpy()     # [1, 4, 768]
diff = np.abs(x - gpt2_x).max()
print(gpt2_x.shape, "largest difference:", diff)
assert x.shape == (1, 4, 768) and diff < 1e-6; print("✅ your [1, 4, 768] block is exactly what GPT-2 uses")

### 1.6 · Word order is now in the numbers
Same word, different position → different row. Try it:

In [ ]:
cat_at_1 = wte[3797] + wpe[1]
cat_at_50 = wte[3797] + wpe[50]
print("same token row, different sums:", np.abs(cat_at_1 - cat_at_50).max().round(4))

### 1.7 · Explore: which words are close?
**Cosine similarity** compares directions: 1 = same direction, 0 = no shared direction, negative = opposite.

✏️ **YOUR TURN:** finish the function. (Hint: `np.linalg.norm(v)` is a vector's length.)

In [ ]:
def nearest(word, k=8):
    i = tok(word)["input_ids"]
    assert len(i) == 1, "pick a word that is a single token (try a leading space)"
    v = wte[i[0]]
    sims = (wte @ v) / (np.linalg.norm(wte, axis=1) * ___)
    best = np.argsort(-sims)[1:k+1]
    return [(tok.decode([j]), round(float(sims[j]), 3)) for j in best]

nearest(" cat")

In [ ]:
assert nearest(" cat")[0][0] == " Cat"; print("✅ now try your own:", nearest(" king"))

### 1.8 · Bonus: what the position rows look like
Neighbouring positions should look alike.

In [ ]:
import matplotlib.pyplot as plt
P = wpe[:64] / np.linalg.norm(wpe[:64], axis=1, keepdims=True)
plt.imshow(P @ P.T, cmap="magma"); plt.colorbar(); plt.title("cosine similarity of positions 0-63"); plt.show()
print("length of row 0:", np.linalg.norm(wpe[0]).round(3), " row 1:", np.linalg.norm(wpe[1]).round(3))

**One document is one string.** A web page or a whole book goes in as one string. The tokenizer knows nothing about lines:
a line break is just the character `\n`, and it becomes a token (ID 198) like any other. One string in, one flat list out, however long.

In [ ]:
doc = "The cat sat on the mat.\nIt was warm.\n\nLater, the dog came in."
doc_ids = tok(doc)["input_ids"]
print(len(doc_ids), doc_ids)
print([tok.decode([i]) for i in doc_ids])
print("decode gives the same text back:", tok.decode(doc_ids) == doc)
print("several strings -> one list each (ragged):", [len(x) for x in tok([doc, "Paris is the capital of France.", "Hello!"])["input_ids"]])

In [ ]:
assert len(doc_ids) == 21 and doc_ids.count(198) == 3 and tok.decode(doc_ids) == doc
print("✅ one document = one string = one flat list; the line breaks are tokens (198)")

### 1.9 · Many texts at once: a batch
A batch is a table of IDs with shape `[B, T]`: B rows, T positions. Texts have different lengths, so the simplest way to run prompts together is to **pad** the short ones and keep an **attention mask** (1 = real, 0 = pad). GPT-2 has no pad token, so we reuse end-of-text (ID 50256).

✏️ **YOUR TURN:** fill the `ids` and `mask` tables.

In [ ]:
prompts = ["The cat sat on", "My dog", "Once upon a time"]
ragged = [tok(p)["input_ids"] for p in prompts]
print(ragged)                     # lists of different lengths

B, T = len(ragged), max(len(r) for r in ragged)
ids_b = np.full((B, T), 50256)    # start with every cell = pad
mask = np.zeros((B, T), dtype=int)
for b, r in enumerate(ragged):
    ids_b[b, :len(r)] = ___
    mask[b, :len(r)] = ___
print(ids_b); print(mask)

In [ ]:
assert ids_b[1].tolist() == [3666, 3290, 50256, 50256] and mask[1].tolist() == [1, 1, 0, 0]; print("✅ a padded [B, T] batch with its mask")

✏️ **YOUR TURN:** embed the whole batch in one line each. `wte[ids_b]` works on a 2-D table of IDs and gives `[B, T, 768]`.
Then add the position rows `[T, 768]`: numpy copies them onto every text for you. That's **broadcasting**.

In [ ]:
X_b = ___                 # token rows for the whole batch
X_b = X_b + ___           # + the position rows for positions 0..T-1
print(X_b.shape)

In [ ]:
with torch.no_grad():
    hs = model(torch.tensor(ids_b), attention_mask=torch.tensor(mask), output_hidden_states=True).hidden_states[0].numpy()
real = mask == 1
print("largest difference on real tokens:", np.abs(X_b - hs)[real].max())
alone = wte[ragged[1]] + wpe[:len(ragged[1])]   # "My dog", the padded one
assert X_b.shape == (3, 4, 768) and np.abs(X_b - hs)[real].max() < 1e-6 and np.allclose(X_b[1, :2], alone)
print("✅ [3, 4] → [3, 4, 768], and row 1 (My dog) inside the batch = run alone")

### 1.10 · Training batches: pack, don't pad
Pretraining joins documents with an end-of-text token and cuts the stream into equal rows, so there's no padding. (Chat fine-tuning often does pad: Episode 6.)

✏️ **YOUR TURN:** cut `stream` into rows of `T_train` tokens (drop the leftover).

In [ ]:
docs = ["The cat sat on the mat.", "Paris is the capital of France.", "Water boils at 100 degrees."]
stream = []
for d in docs:
    stream += tok(d)["input_ids"] + [50256]
print(len(stream), "tokens in the stream")

T_train = 6
n_rows = len(stream) // T_train
rows = np.array(stream[:___]).reshape(n_rows, T_train)
for r in rows:
    print([tok.decode([i]) for i in r])

In [ ]:
assert rows.shape == (3, 6) and rows[1, 1] == 50256; print("✅ packed [3, 6]. Notice row 1 starts mid-sentence and holds two documents")

### 1.11 · Matrix multiplication with 3 dimensions
🎬 **Video walkthrough:** [Deep dive D1.1 · 3-D Matrix Multiplication, Finally Visual](https://youtu.be/25GaBn2QYW4)

Rule: `[B, T, K] @ [K, N] = [B, T, N]`. Every sheet is multiplied by the **same** W.

✏️ **YOUR TURN:** compute `Y` three ways and check they agree.

In [ ]:
X = np.array([[[1, 0, 2, -1], [0, 1, 1, 2], [2, -1, 0, 1]],
              [[-1, 2, 0, 1], [1, 1, -1, 0], [0, 2, 1, -2]]])   # [2, 3, 4]
W = np.array([[1, 0], [2, 1], [0, -1], [1, 2]])                  # [4, 2]

Y_direct = ___                                       # just use @
Y_loop   = np.stack([___ for b in range(2)])         # sheet by sheet
Y_flat   = (X.reshape(6, 4) @ W).reshape(___)        # flatten, multiply, cut back
print(Y_direct.shape); print(Y_direct)

In [ ]:
assert np.array_equal(Y_direct, Y_loop) and np.array_equal(Y_direct, Y_flat) and Y_direct[0, 0, 0] == 0
print("✅ all three agree: the extra dimension just comes along")

### 1.12 · The lookup, by hand
🎬 **Video walkthrough:** [Deep dive D1.1 · 3-D Matrix Multiplication, Finally Visual](https://youtu.be/25GaBn2QYW4)

`wte` is GPT-2's **token table**, `wpe` its **position table**, `ids` the `[B, T]` token IDs. For every cell of the ID table
(batch row `b`, position `t`), the step works on **whole rows**:

`x[b, t] = wte[ ids[b, t] ] + wpe[ t ]`

Copy the token table's row whose number is the ID, copy the position table's row whose number is the position, and add the two rows.
The ID only says *which* row: its value never goes into the sum. The position row depends on `t` only, so every batch row gets the same one.

✏️ **YOUR TURN:** write that rule inside the two loops, on a tiny made-up model (5 tokens, rows of 3 numbers).

In [ ]:
E = np.array([[2, 0, 1], [1, 3, 0], [0, 1, 4], [3, 2, 2], [1, 1, 0]])   # token table [5, 3]
P = np.array([[0, 1, 0], [1, 0, 0], [0, 0, 1]])                         # position table [3, 3]
ids_toy = np.array([[2, 4], [3, 2]])                                     # [B, T] = [2, 2]

B_, T_ = ids_toy.shape
x_loop = np.zeros((B_, T_, 3), dtype=int)
for b in range(B_):
    for t in range(T_):
        x_loop[b, t] = ___        # 2 of 3 indices: a whole row of 3 numbers (same as x_loop[b, t, :])
        print(f"b={b} t={t}  ID {ids_toy[b, t]} -> token row {E[ids_toy[b, t]]} + position row {P[t]} = {x_loop[b, t]}")

In [ ]:
x_fast = E[ids_toy] + P[:T_]     # lookup [2, 2] -> [2, 2, 3], plus P[:2] [2, 3] broadcast onto both batch rows
x_onehot = np.eye(5, dtype=int)[ids_toy] @ E + P[:T_]   # one-hot [2, 2, 5] @ [5, 3]
assert np.array_equal(x_loop, x_fast) and np.array_equal(x_loop, x_onehot) and x_loop[0, 0].tolist() == [0, 2, 4]
print("✅ loops, lookup and one-hot all give the same [2, 2, 3]")

**What `x[b, t]` picks out.** `x` has 3 dimensions; each index you give narrows it down, and the ones you leave off the end are taken in full.

In [ ]:
print("x_fast          ", x_fast.shape)        # [2, 2, 3]: the whole block
print("x_fast[1]       ", x_fast[1].shape)     # one batch row: [2, 3]
print("x_fast[1, 0]    ", x_fast[1, 0], x_fast[1, 0].shape)        # one token's row: [3]
print("x_fast[1, 0, :] ", x_fast[1, 0, :])                         # the same row, with ':' = all of this dimension
print("x_fast[1, 0, 2] ", x_fast[1, 0, 2])                         # one number

**How `E[ids]` picks rows.** Indexing a table with numbers hands back those rows, in the shape of the index. Grow the index step by step:

In [ ]:
print("E[2]       ", E[2], "shape", E[2].shape)                 # one number  -> one row
print("E[[2, 4]]  ", E[[2, 4]].tolist(), "shape", E[[2, 4]].shape)  # a list     -> those rows, stacked
print("E[ids_toy] shape", E[ids_toy].shape)                       # a [2, 2] table -> a row for every cell

# PyTorch does exactly the same, three ways:
Et, it = torch.tensor(E), torch.tensor(ids_toy)
emb = torch.nn.Embedding.from_pretrained(Et.float())             # a layer holding the table, like GPT-2's wte
print(Et[it].shape, torch.equal(emb(it), Et[it].float()), torch.equal(torch.index_select(Et, 0, it.reshape(-1)).reshape(2, 2, 3), Et[it]))

**From `[B, T]` to a one-hot block `[B, T, V]`.** Every ID is replaced by a row of V numbers (one 1, the rest 0), which adds a
new last dimension of size V. Three ways, same block:

In [ ]:
V = 5
ids_b1 = np.array([[2, 4]])                                   # [1, 2]
oh_loop = np.zeros(ids_b1.shape + (V,), dtype=int)            # zeros, [1, 2, 5]
for b in range(ids_b1.shape[0]):
    for t in range(ids_b1.shape[1]):
        oh_loop[b, t, ids_b1[b, t]] = 1                       # a 1 at position ids[b, t]
oh_eye = np.eye(V, dtype=int)[ids_b1]                         # row i of the identity table is the one-hot row for ID i
oh_torch = torch.nn.functional.one_hot(torch.tensor(ids_b1), V)
print(ids_b1.shape, "->", oh_loop.shape, oh_loop.tolist())
print("batch of 2:", ids_toy.shape, "->", np.eye(V, dtype=int)[ids_toy].shape)
assert np.array_equal(oh_loop, oh_eye) and np.array_equal(oh_loop, oh_torch.numpy())
print("✅ [B, T] -> [B, T, V]: loop, np.eye and F.one_hot give the same block")

**One-hot, row by row.** A one-hot row times E is a weighted sum of E's rows: each one-hot number multiplies one whole row,
then the products are added. Only the row that meets the 1 survives. One token first, then a batch of 1, then a batch of 2.

In [ ]:
oh = np.eye(5, dtype=int)                  # row i of this is the one-hot row for ID i
w = oh[2]                                  # ID 2 -> [0 0 1 0 0]
parts = [w[v] * E[v] for v in range(5)]    # each one-hot number multiplies one whole row of E
for v in range(5):
    print(f"{w[v]} x row {v} {E[v]} = {parts[v]}")
print("add them:", sum(parts), "  = row 2:", E[2], "  = w @ E:", w @ E)

ids_b1 = np.array([[2, 4]])                # a batch of 1 text with 2 tokens: [1, 2]
print("batch of 1:", oh[ids_b1].shape, "@", E.shape, "=", (oh[ids_b1] @ E).shape, (oh[ids_b1] @ E).tolist())
print("batch of 2:", oh[ids_toy].shape, "@", E.shape, "=", (oh[ids_toy] @ E).shape, (oh[ids_toy] @ E).tolist())
assert np.array_equal(sum(parts), E[2]) and np.array_equal(oh[ids_b1] @ E, E[ids_b1]) and np.array_equal(oh[ids_toy] @ E, E[ids_toy])
print("✅ one-hot @ E = the lookup, for one token, a batch of 1 and a batch of 2")

**Positions are looked up by position number, not by token ID.** The position numbers only depend on the length T.

✏️ **YOUR TURN:** make the position numbers for our batch and look them up.

In [ ]:
positions = ___            # the numbers 0, 1, ..., T-1 for our batch from 1.9 (hint: np.arange)
pos_rows_b = wpe[positions]
print(positions, pos_rows_b.shape)

In [ ]:
assert positions.tolist() == [0, 1, 2, 3] and np.array_equal(pos_rows_b, wpe[:4])
print("✅ [T] position numbers → [T, 768] rows, shared by every row of the batch")

**What GPT-2's own code does:** `nn.Embedding`, an index lookup. It never builds one-hot rows, but it would get the same numbers.

In [ ]:
import torch.nn.functional as F
W = model.transformer.wte.weight
ids_t = torch.tensor(ids_b)
with torch.no_grad():
    a = model.transformer.wte(ids_t)                       # what GPT-2 runs
    b = W[ids_t]                                           # plain indexing
    c = F.one_hot(ids_t, 50257).to(W.dtype) @ W            # one-hot [3, 4, 50257] @ [50257, 768]
print(a.shape, torch.equal(a, b), torch.equal(a, c))
print(f"one-hot costs {50257 * 768:,} multiply-adds per token; the lookup copies 768 numbers")

### 1.13 · The real data: TinyStories
So far our text was made up. From here on we also use the dataset we'll **train on** in the training chapters:
[TinyStories](https://huggingface.co/datasets/roneneldan/TinyStories) (Eldan & Li, 2023). It's about 2.1 million short stories,
generated by GPT-3.5 and GPT-4, that only use words "a typical 3 to 4-year-olds usually understand". The paper shows that models
with **under 10 million parameters** trained on it write fluent, consistent stories, which makes it a good fit for a free Colab GPU.

Every chapter will do its new step on this data too. Today: text → token IDs → one long stream → a file → a packed `[B, T]` batch → `[B, T, 768]`.

The data comes as 4 training files (~250 MB each) and 1 validation file (10 MB). We use **the first training file**, about a quarter of
the stories, for this chapter *and* for training later: on a free Colab GPU, the training time runs out long before this much data
does. Chapter 1 turns it into `train.bin` once; the training chapters reuse that file.

In [ ]:
import pandas as pd
from huggingface_hub import hf_hub_download

TS_REPO = "roneneldan/TinyStories"
TS_FILES = {"train": ["data/train-00000-of-00004-2d5a1467fff1081b.parquet"],     # 1 of the 4 training files
            "val":   ["data/validation-00000-of-00001-869c898b519ad725.parquet"]}
stories = {split: [s for f in files for s in pd.read_parquet(hf_hub_download(TS_REPO, f, repo_type="dataset"))["text"]]
           for split, files in TS_FILES.items()}
for split in stories:
    print(f"{split}: {len(stories[split]):,} stories")
print("\n--- validation story 0 ---\n" + stories["val"][0])

**One story → IDs.** Exactly like the document in 1.8: one story is one string, one flat list. To join stories into one stream
(packing, 1.10) we put the end-of-text token after each one, so the model can tell where a story stops.

✏️ **YOUR TURN:** add the end-of-text ID (50256, also `tok.eos_token_id`) after the story's IDs.

In [ ]:
story_ids = tok(stories["val"][0])["input_ids"] + ___
print(len(story_ids), "tokens:", story_ids[:10], "...", story_ids[-3:])

In [ ]:
assert story_ids[-1] == 50256 and story_ids.count(50256) == 1 and tok.decode(story_ids[:-1]) == stories["val"][0]
print("✅ one story = one string = one list, + end-of-text")

**All stories → one long stream, written to a file.** The same thing for every story: IDs + end-of-text, all joined. But we
don't build the whole stream in memory first. Hundreds of thousands of stories give over 100 million IDs, and as Python lists that's
many gigabytes, which can crash a free Colab session. So we go **10,000 stories at a time**: tokenize them (`tok(list_of_texts)` does a
whole list in one fast call), turn the IDs into a compact numpy array, and append it to the end of the file. That's how nanoGPT's data
scripts do it too.

This takes a few minutes for the training file. (You may see a warning that a sequence is longer than 1,024 tokens: harmless. Some
stories are long, and the model never sees a whole story at once; it only sees rows of T tokens cut from the stream.)

In [ ]:
import json, time
DATA = os.path.join(WORK, "data", "tinystories")
os.makedirs(DATA, exist_ok=True)

def write_stream(texts, path, piece=10_000):
    n_ids, t0 = 0, time.time()
    with open(path, "wb") as f:
        for i in range(0, len(texts), piece):
            pieces = [np.array(ids + [tok.eos_token_id], dtype=np.uint16) for ids in tok(texts[i : i + piece])["input_ids"]]
            block = np.concatenate(pieces)       # this piece's IDs, 2 bytes each (uint16: why, in the next cell)
            block.tofile(f)                      # append the raw bytes to the file
            n_ids += len(block)
            print(f"\r  {min(i + piece, len(texts)):,} / {len(texts):,} stories · {n_ids:,} IDs · {time.time() - t0:.0f} s", end="")
    print()
    return n_ids

for split in ["val", "train"]:
    print(split)
    write_stream(stories[split], os.path.join(DATA, split + ".bin"))
    json.dump({"files": TS_FILES[split]}, open(os.path.join(DATA, split + ".json"), "w"))   # a note on what's inside, for the toolkit (1.14)

### What is a `.bin` file?
We saved each stream to a file, `train.bin` and `val.bin` (the names nanoGPT uses). **.bin just means binary**: the file holds the
numbers' raw bytes, nothing else. No commas, no spaces, no digits as text, and no header saying what's inside.

- **What's actually stored:** every token ID is a `uint16`, a whole number in **2 bytes** (16 bits), so 0 to 65,535. GPT-2's largest ID
  is 50,256, so it fits. Numpy's default for whole numbers, `int64`, would take 8 bytes each: 4× bigger.
- **Hexadecimal, or 0s and 1s?** Neither, and both. The disk stores bits; hex and 0/1 are only two ways **we print** the same bytes.
  ID 464 = 1·256 + 208, so it's the two bytes 208 and 1. Printed in hex: `d0 01`; in bits: `11010000 00000001`. All the same thing.
- **Byte order:** numpy writes the machine's own order, and on PCs, Macs and Colab that's low byte first ("little-endian").
  That's why it's `d0 01`, not `01 d0`.
- **A text file would be bigger and slower:** `464` as text is the 3 characters `4`, `6`, `4` (3 bytes) plus a separator, and it has to
  be turned back into a number every time it's read. In binary the computer just copies 2 bytes.
- **Reading it back, you must say the type:** the file doesn't know it's `uint16`. Read it as `int32` and you get nonsense (try it below).

In [ ]:
val_ids = np.fromfile(os.path.join(DATA, "val.bin"), dtype=np.uint16)   # read the whole (small) file back
raw = open(os.path.join(DATA, "val.bin"), "rb").read(6)                 # the first 6 bytes of the file = the first 3 IDs
print("first 3 IDs:      ", val_ids[:3].tolist())
print("bytes as numbers: ", list(raw))
print("same bytes in hex:", raw.hex(" "))
print("same bytes as bits:", " ".join(f"{b:08b}" for b in raw))
print(f"\nval.bin: {len(val_ids):,} IDs, {os.path.getsize(os.path.join(DATA, 'val.bin')):,} bytes on disk")
as_text = " ".join(map(str, val_ids.tolist()))
print(f"the same IDs as a text file would be {len(as_text):,} bytes")

✏️ **YOUR TURN:** (1) predict the size of `val.bin` in bytes from the number of tokens. (2) Rebuild the first ID from its two bytes
(low byte first: `low + 256 × high`).

In [ ]:
expected_bytes = ___
first_id = ___
print(expected_bytes, first_id)

In [ ]:
assert expected_bytes == os.path.getsize(os.path.join(DATA, "val.bin")) and first_id == val_ids[0]
print("✅ a .bin file is just the numbers' bytes: 2 per token, low byte first")

**Reading the big one.** `train.bin` is a few hundred MB. `np.memmap` opens it *without* loading it: numpy reads bytes from disk
only for the parts you touch. It's how nanoGPT reads its data. You give the type yourself:

In [ ]:
data = np.memmap(os.path.join(DATA, "train.bin"), dtype=np.uint16, mode="r")
print(f"train.bin: {len(data):,} IDs, {os.path.getsize(os.path.join(DATA, 'train.bin')) / 1e6:.0f} MB · largest ID: {data.max()}")
print(f"{len(stories['train']):,} stories, about {len(data) / len(stories['train']):.0f} tokens per story")
first = tok(stories["train"][0])["input_ids"] + [tok.eos_token_id]
print("starts with story 0 + end-of-text:", data[:len(first)].tolist() == first)
print("read as uint16:", data[:4].tolist(), "→", repr(tok.decode(data[:4].tolist())))
wrong = np.fromfile(os.path.join(DATA, "train.bin"), dtype=np.int32, count=4)
print("read as int32: ", wrong.tolist(), " ← nonsense: each 'number' glued two IDs' bytes together")

**A packed batch, from the real data.** Cut the stream into rows of T tokens, as in 1.10. B = 8 rows of T = 64 tokens.

✏️ **YOUR TURN:** take the first B × T IDs and reshape them into `[B, T]`.

In [ ]:
B_ts, T_ts = 8, 64
batch = np.asarray(data[:B_ts * T_ts]).reshape(___).astype(np.int64)   # int64: the type PyTorch wants for IDs
print(batch.shape)
for r in batch[:3]:
    print(repr(tok.decode(r.tolist())))

In [ ]:
assert batch.shape == (8, 64) and batch[1, 0] == data[64]
print("✅ a packed [8, 64] batch of real training data. Rows start mid-sentence, and <|endoftext|> marks where stories meet")

Training code doesn't cut rows in order: every step it picks **B random starting points** in the stream and takes T tokens from each
(nanoGPT's `get_batch`). Same shape, a fresh mix of stories every step. Then chapter 1's lookup turns it into vectors:

In [ ]:
def get_batch(data, B, T):
    starts = np.random.randint(0, len(data) - T, size=B)       # B random places in the stream
    return np.stack([data[s : s + T] for s in starts]).astype(np.int64)

xb = get_batch(data, B_ts, T_ts)
X_ts = wte[xb] + wpe[:T_ts]                                   # [8, 64] -> [8, 64, 768]
with torch.no_grad():
    hs = model(torch.tensor(xb), output_hidden_states=True).hidden_states[0].numpy()
print(xb.shape, "->", X_ts.shape, " largest difference from GPT-2:", np.abs(X_ts - hs).max())
assert X_ts.shape == (8, 64, 768) and np.abs(X_ts - hs).max() < 1e-5
print("✅ real training batch [8, 64] -> [8, 64, 768], exactly what GPT-2 would get")

### 1.14 · Chapter 1 toolkit
Later chapters start here. Run this cell to get chapter 1's results without redoing the exercises: `embed()`, and `tinystories()` +
`get_batch()` for the real data. `tinystories()` reuses `train.bin` / `val.bin` from 1.13 if they're there; if not (a new session
without Drive), it downloads and tokenizes again, which takes a few minutes.

In [ ]:
import json
import pandas as pd
from huggingface_hub import hf_hub_download

TS_REPO = "roneneldan/TinyStories"
TS_FILES = {"train": ["data/train-00000-of-00004-2d5a1467fff1081b.parquet"],
            "val":   ["data/validation-00000-of-00001-869c898b519ad725.parquet"]}

def embed(ids_b):
    """Token IDs [B, T] -> GPT-2's input to layer 1, [B, T, 768]. One sentence is [1, T]."""
    ids_b = np.asarray(ids_b)
    T = ids_b.shape[-1]
    assert T <= wpe.shape[0], "GPT-2 has position rows for 1,024 tokens only"
    return wte[ids_b] + wpe[:T]

def tinystories(split):
    """TinyStories as one long stream of GPT-2 token IDs (stories joined by end-of-text), opened from <split>.bin with memmap.
    split: "train" or "val". Builds the file only if it isn't there yet (download, then tokenize 10,000 stories at a time)."""
    folder = os.path.join(WORK, "data", "tinystories")
    os.makedirs(folder, exist_ok=True)
    bin_path, note_path = os.path.join(folder, split + ".bin"), os.path.join(folder, split + ".json")
    want = {"files": TS_FILES[split]}
    if not (os.path.exists(bin_path) and os.path.exists(note_path) and json.load(open(note_path)) == want):
        print(f"building {split}.bin (first time only) ...")
        texts = [s for f in TS_FILES[split] for s in pd.read_parquet(hf_hub_download(TS_REPO, f, repo_type="dataset"))["text"]]
        with open(bin_path, "wb") as out:
            for i in range(0, len(texts), 10_000):
                ids = tok(texts[i : i + 10_000])["input_ids"]
                np.concatenate([np.array(x + [tok.eos_token_id], dtype=np.uint16) for x in ids]).tofile(out)   # 2 bytes per ID
        json.dump(want, open(note_path, "w"))
    return np.memmap(bin_path, dtype=np.uint16, mode="r")

def get_batch(data, B, T):
    """B random rows of T tokens from the stream -> IDs [B, T] (int64)."""
    starts = np.random.randint(0, len(data) - T, size=B)
    return np.stack([data[s : s + T] for s in starts]).astype(np.int64)

wte = model.transformer.wte.weight.detach().numpy()
wpe = model.transformer.wpe.weight.detach().numpy()
print(embed([[464, 3797, 3332, 319]]).shape)
train_data, val_data = tinystories("train"), tinystories("val")
print(f"TinyStories: {len(train_data):,} train tokens, {len(val_data):,} val tokens ·", embed(get_batch(train_data, 8, 64)).shape)

### 1.15 · Build our own GPT: TinyGPT, part 1
Everything so far took GPT-2's tables apart. Now we start **our own model, TinyGPT**, in PyTorch. Its weights start **random**, and in the
training chapters we train them on our TinyStories file. TinyGPT grows one piece per chapter, and this section is always the last one:

| Chapter | TinyGPT gains |
|---|---|
| 1 (today) | the token table `wte` + the position table `wpe` |
| 2-4 | attention, a full block, all the blocks + the final layer that scores every next word |
| 5-8 | the loss, then training it from scratch, and saving it |
| later | writing text, a modern version (RMSNorm, RoPE, SwiGLU, GQA), a mixture-of-experts version, serving it |

**One class, any size.** The code is written once; a few numbers decide how big it is: `n_layer` blocks, each token's vector `n_embd`
numbers wide, `n_head` attention heads (chapter 2), and `block_size`, the longest text it can read. GPT-2 small is 12 layers × 768.
TinyGPT's default is 6 layers × 384, small enough to train on a free Colab GPU.

In [ ]:
import torch.nn as nn
from dataclasses import dataclass

@dataclass
class TinyGPTConfig:
    vocab_size: int = 50257   # GPT-2's tokenizer, so we keep its 50,257 IDs
    block_size: int = 256     # longest text TinyGPT can read (positions 0..255)
    n_layer: int = 6          # blocks stacked (they arrive in chapters 2-4)
    n_head: int = 6           # attention heads (chapter 2)
    n_embd: int = 384         # width of each token's vector

class TinyGPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.wte = nn.Embedding(cfg.vocab_size, cfg.n_embd)   # token table   [vocab_size, n_embd]
        self.wpe = nn.Embedding(cfg.block_size, cfg.n_embd)   # position table [block_size, n_embd]
        # start random, the way OpenAI's GPT-2 code does: small normal numbers, std 0.02 (tokens) and 0.01 (positions)
        nn.init.normal_(self.wte.weight, mean=0.0, std=0.02)
        nn.init.normal_(self.wpe.weight, mean=0.0, std=0.01)
        # coming next: chapter 2 attention, chapter 3 a full block, chapter 4 n_layer blocks + final LayerNorm + next-word scores

    def forward(self, idx):
        """Token IDs [B, T] -> vectors [B, T, n_embd]. (From chapter 4 on: next-word scores.)"""
        B, T = idx.shape
        assert T <= self.cfg.block_size, f"TinyGPT reads at most {self.cfg.block_size} tokens"
        pos = ___                        # the position numbers 0..T-1, as a torch tensor on idx's device
        x = self.wte(idx) + ___          # token rows [B, T, n_embd] + position rows [T, n_embd] (broadcast)
        return x

✏️ **YOUR TURN:** fill the two blanks above (Hint: `torch.arange(T, device=idx.device)`; then the same lookup as 1.12, done by
`nn.Embedding`), run the cell, then make a TinyGPT and feed it a real TinyStories batch.

In [ ]:
torch.manual_seed(0)
tiny = TinyGPT(TinyGPTConfig())
n_params = sum(p.numel() for p in tiny.parameters())
print(f"TinyGPT so far: {n_params:,} weights, all random")
for name, p in tiny.named_parameters():
    print(f"  {name:12s} {list(p.shape)}  std {p.std().item():.4f}")

xb = torch.tensor(get_batch(train_data, 8, 64))   # a real [8, 64] batch (toolkit 1.14)
out = tiny(xb)
print(list(xb.shape), "->", list(out.shape))

In [ ]:
assert out.shape == (8, 64, 384) and n_params == 50257 * 384 + 256 * 384
print("✅ TinyGPT turns a real [8, 64] TinyStories batch into [8, 64, 384] vectors")

**Is our code right? A one-time test against GPT-2.** Our random TinyGPT can't be compared with anything yet. But because the size is
just a setting, we can build a *test copy* at GPT-2's exact size (768 wide, 1,024 positions), copy GPT-2's two tables into it, and check
it gives exactly what GPT-2 sends into its first layer. If it does, our code is right. Then we throw the test copy away: TinyGPT keeps its
own random weights. (GPT-2's tables could never go into TinyGPT itself: `[50257, 768]` doesn't fit into `[50257, 384]`.)

In [ ]:
check = TinyGPT(TinyGPTConfig(block_size=1024, n_layer=12, n_head=12, n_embd=768))   # GPT-2 small's sizes
with torch.no_grad():
    check.wte.weight.copy_(model.transformer.wte.weight)
    check.wpe.weight.copy_(model.transformer.wpe.weight)
    ours = check(xb)
    gpt2 = model(xb, output_hidden_states=True).hidden_states[0]
diff = (ours - gpt2).abs().max().item()
print(list(ours.shape), "largest difference from GPT-2:", diff)
assert diff < 1e-5
del check                                                   # only a test: TinyGPT stays random
print("✅ our TinyGPT code computes exactly what GPT-2 computes. Test copy deleted; TinyGPT keeps its random weights")

**What random weights mean.** TinyGPT's token table knows nothing yet. In GPT-2's trained table (1.7) the words closest to " cat"
were " Cat", " cats" and other animals. In TinyGPT's random table the closest words are just noise. Training is what gives the rows
meaning, and every chapter from now on adds the parts that training will tune.

In [ ]:
def nearest_in(table, word, k=6):
    v = table[tok(word)["input_ids"][0]]
    sims = (table @ v) / (table.norm(dim=1) * v.norm())
    return [tok.decode([j]) for j in sims.argsort(descending=True)[1:k+1].tolist()]

with torch.no_grad():
    print("GPT-2 (trained): ", nearest_in(model.transformer.wte.weight, " cat"))
    print("TinyGPT (random):", nearest_in(tiny.wte.weight, " cat"))

#### Chapter 1 solutions
```python
top_left   = np.sum(A[0] * B[:, 0])
ids        = tok(text, return_tensors="np")["input_ids"]   # [1, 4]
tok_rows   = wte[ids]                                  # [1, 4, 768]
onehot[0, t, i] = 1.0
via_matmul = onehot @ wte
pos_rows   = wpe[:4]          # [4, 768]; or wpe[np.arange(ids.shape[1])]
x          = tok_rows + pos_rows
# nearest(): ... * np.linalg.norm(v)
ids_b[b, :len(r)] = r
mask[b, :len(r)]  = 1
X_b = wte[ids_b]
X_b = X_b + wpe[:T]
rows = np.array(stream[:n_rows * T_train]).reshape(n_rows, T_train)
Y_direct = X @ W
Y_loop = np.stack([X[b] @ W for b in range(2)])
Y_flat = (X.reshape(6, 4) @ W).reshape(2, 3, 2)
x_loop[b, t] = E[ids_toy[b, t]] + P[t]
positions = np.arange(ids_b.shape[1])
story_ids = tok(stories["val"][0])["input_ids"] + [50256]      # or [tok.eos_token_id]
expected_bytes = len(val_ids) * 2
first_id = raw[0] + 256 * raw[1]
batch = np.asarray(data[:B_ts * T_ts]).reshape(B_ts, T_ts).astype(np.int64)
# TinyGPT.forward
pos = torch.arange(T, device=idx.device)
x = self.wte(idx) + self.wpe(pos)
```
**Next chapter:** these rows start talking to each other: attention by hand, with the causal mask, on "The cat sat on" and on a real
TinyStories batch. And TinyGPT gets its first attention layer.

---
## Chapter 2 · Attention by hand
The rows from chapter 1 start **talking to each other**. You'll run one real GPT-2 attention head (layer 0, head 4) on
**"The cat sat on"** step by step in plain numpy: first the LayerNorm that comes right before it, then queries, keys and values, the `[1, 4, 4]` score table, dividing by 8, the causal mask,
softmax, and the weighted mix. Every step is checked against GPT-2's own numbers. Then the same head on a real **TinyStories** batch
`[8, 64]`, and **TinyGPT** gets its first attention head. Needs **Setup** and the **chapter 1 toolkit** (1.14).

### 2.0 · Get ready
Chapter 1's toolkit (1.14) gave us `embed()`, `tinystories()` and `get_batch()`. Run it first if this is a new session.

One more switch. Setup loaded GPT-2 with PyTorch's **fast attention**, which never keeps the table of attention weights, so GPT-2
can't show them to us. For this chapter we switch it to the plain ("eager") version: the same math, step by step, and it can hand
back its weights with `output_attentions=True`. The results are the same either way.

In [ ]:
import math
import matplotlib.pyplot as plt

model.set_attn_implementation("eager")     # the plain attention code, so GPT-2 can show its weights
blk = model.transformer.h[0]               # layer 0: the first of GPT-2's 12 blocks
print(blk)

### 2.1 · Step 0: LayerNorm, every row to the same size
Chapter 1 ended with `x`, shape `[1, 4, 768]`: token row + position row for each token. Attention doesn't read `x` directly. The first
thing inside each layer is a **LayerNorm** (GPT-2 calls it `ln_1`). Look at the rows first: their **spread** (standard deviation, roughly
how far the 768 numbers sit from their average) is quite different.

In [ ]:
text = "The cat sat on"
ids = tok(text, return_tensors="np")["input_ids"]          # [1, 4]
x = embed(ids)                                              # [1, 4, 768]   (chapter 1)
pieces = [tok.decode([i]) for i in ids[0]]
print(pieces, "x", x.shape)
print("spread of each row of x:", x.std(axis=-1).round(3))
print("length of position rows 0-3:", np.linalg.norm(wpe[:4], axis=1).round(2), "<- position 0 is unusually long")

LayerNorm puts every row on the same scale, **one row at a time** (rows never mix). The recipe, on one whole row:

| step | formula |
|---|---|
| 1. mean | `μ` = mean of the row's 768 numbers |
| 2. variance | `σ²` = mean of `(x − μ)²` |
| 3. normalize | `z = (x − μ) / √(σ² + ε)`: now mean 0, spread 1. `ε` = 1e-5 stops a division by 0 |
| 4. scale and shift | `h = z · γ + β`, with `γ` (gamma) and `β` (beta) 768 **learned** numbers each |

PyTorch calls `γ` `.weight` and `β` `.bias`. In numpy, `axis=-1` means "along the 768 numbers of each row", and `keepdims=True` keeps
`μ` as `[1, 4, 1]`, so `x - mu` broadcasts each row's own mean across that row.

✏️ **YOUR TURN:** finish `layer_norm` (the variance line).

In [ ]:
gamma = blk.ln_1.weight.detach().numpy()     # γ [768]
beta  = blk.ln_1.bias.detach().numpy()       # β [768]
print("gamma", gamma.shape, "from", gamma.min().round(3), "to", gamma.max().round(3), "· eps", blk.ln_1.eps)

def layer_norm(x, gamma, beta, eps=1e-5):
    """LayerNorm over the last axis: each row -> mean 0, spread 1, then * gamma + beta"""
    mu = x.mean(axis=-1, keepdims=True)                  # [..., 1]
    var = ___                                            # [..., 1]: the mean of (x - mu) squared
    z = (x - mu) / np.sqrt(var + eps)
    return z * gamma + beta

h = layer_norm(x, gamma, beta)               # [1, 4, 768]: what attention reads
print("h", h.shape, "· spread of each row of h:", h.std(axis=-1).round(3))
print("h, first 4 numbers of each row:\n", h[0, :, :4].round(4))

In [ ]:
gpt2_h = blk.ln_1(torch.tensor(x)).detach().numpy()
print("largest difference from GPT-2's own ln_1:", np.abs(h - gpt2_h).max())
assert np.abs(h - gpt2_h).max() < 1e-5
print("✅ your LayerNorm is GPT-2's ln_1")

**Try it:** run `layer_norm(x, np.ones(768), np.zeros(768))` (γ = 1, β = 0, how training starts) and print each row's mean and
spread. You should get 0 and 1 for every row. GPT-2's trained γ (mostly around 0.18) then shrinks them again.

### 2.2 · Cut one head's tables out of `c_attn`
GPT-2 keeps layer 0's query, key and value tables for **all 12 heads** in one block, `c_attn`:

| | shape | what's inside |
|---|---|---|
| `c_attn.weight` | `[768, 2304]` | columns 0–767 make queries, 768–1535 keys, 1536–2303 values |
| `c_attn.bias` | `[2304]` | one number added to each column |

Each block of 768 columns is 12 heads × 64 columns side by side, so **head 4** owns columns 256–319 of each block.
(Hugging Face's `Conv1D` stores the table as `[in, out]` = `[768, 2304]`, like OpenAI's original; we use it as `h @ W + b`.)

In [ ]:
Wc = blk.attn.c_attn.weight.detach().numpy()     # [768, 2304]
bc = blk.attn.c_attn.bias.detach().numpy()       # [2304]
HEAD, HS = 4, 64                                 # the head we follow, and its width (768 / 12)

def cols(kind, head):
    """the 64 columns of c_attn that belong to one head. kind: "q", "k" or "v"."""
    start = {"q": 0, "k": 768, "v": 1536}[kind] + head * HS
    return slice(start, start + HS)

Wq, bq = Wc[:, cols("q", HEAD)], bc[cols("q", HEAD)]
Wk, bk = Wc[:, cols("k", HEAD)], bc[cols("k", HEAD)]
Wv, bv = Wc[:, ___], bc[cols("v", HEAD)]
print("Wq", Wq.shape, "bq", bq.shape, "· Wv", Wv.shape)

✏️ **YOUR TURN:** fill the blank so `Wv` is head 4's value columns. Then check:

In [ ]:
assert Wv.shape == (768, 64) and np.array_equal(Wv[:, 0], Wc[:, 1536 + 256])
print("✅ head 4's value table is columns 1792..1855 of c_attn")

### 2.3 · Step 1: every row becomes a query, a key and a value
For each token, on whole rows: `Q[b, t] = h[b, t] @ Wq + bq`, `[768] @ [768, 64] + [64] → [64]`. The code does all 4 rows at once with
the 3-D rule from chapter 1 (the last dimension meets the second-to-last), and the bias is broadcast to every row.

| | used when this token… |
|---|---|
| query `Q` | is **looking**: which earlier tokens matter to me? |
| key `K` | is **being looked at**: it's compared with the queries |
| value `V` | hands its information over to whoever looks at it |

In [ ]:
Q = ___                      # [1, 4, 64]
K = h @ Wk + bk
V = h @ Wv + bv
print("Q", Q.shape, "K", K.shape, "V", V.shape)
print("Q, first 4 numbers of each row:\n", Q[0, :, :4].round(4))

In [ ]:
assert Q.shape == (1, 4, 64) and np.allclose(Q[0, 3], h[0, 3] @ Wq + bq, atol=1e-5)
print("✅ 4 queries of 64 numbers, one per token")

### 2.4 · Step 2: compare every query with every key
How much should token `i` look at token `j`? GPT-2's answer is the **dot product** of `i`'s query and `j`'s key: multiply the 64 pairs
and add them up. Big and positive = look a lot.

✏️ **YOUR TURN:** compute one score by hand: the query of `" on"` (row 3) with the key of `" sat"` (row 2).

In [ ]:
one_score = ___              # Hint: multiply Q[0, 3] and K[0, 2] number by number, then np.sum
print(round(float(one_score), 4))

Now all of them at once. `K.transpose(0, 2, 1)` turns `K` `[1, 4, 64]` into `[1, 64, 4]` (keys become columns), so
`Q @ Kᵀ` is `[1, 4, 64] @ [1, 64, 4] → [1, 4, 4]`: one score for every (query row, key column) pair.

In [ ]:
scores = Q @ ___             # [1, 4, 4]
print("scores", scores.shape)
print(scores[0].round(2))

In [ ]:
assert np.isclose(one_score, scores[0, 3, 2], atol=1e-4)
print("✅ row 3, column 2 of the table is exactly your hand-made dot product")

### 2.5 · Step 3: divide by 8
Every score is divided by √64 = 8, the square root of the query/key size (the paper calls it `d_k`). Why? A dot product of 64 random
pairs spreads widely (its spread grows like √64), and softmax (next steps) turns big gaps into "all on one token". Dividing by 8 brings
the spread back to about 1. Try it on random vectors:

In [ ]:
rng = np.random.default_rng(0)
qr, kr = rng.standard_normal((100_000, 64)), rng.standard_normal((100_000, 64))
dots = (qr * kr).sum(axis=1)
print("spread of raw dot products:", dots.std().round(2), "· after / 8:", (dots / 8).std().round(3))

scaled = scores / math.sqrt(HS)       # / 8
print(scaled[0].round(4))

### 2.6 · Step 4: the causal mask
GPT-2 learns to predict the next word **at every position at once**, so a token must never look at the tokens after it, or it could
just read the answer. The mask keeps the lower triangle (a token looks at itself and earlier tokens) and sets everything above the
diagonal to −∞, which softmax turns into exactly 0%.

In [ ]:
T = ids.shape[1]
keep = np.tril(np.ones((T, T), dtype=bool))      # [T, T]: True on and below the diagonal
print(keep.astype(int))
masked = np.where(keep, scaled, ___)            # [1, 4, 4]; the [T, T] mask is broadcast over the batch
print(masked[0].round(2))

In [ ]:
assert np.isneginf(masked[0, 0, 1]) and masked[0, 3, 0] == scaled[0, 3, 0]
print("✅ above the diagonal is -inf, the rest is untouched")

### 2.7 · Step 5: softmax turns each row into percentages
For each score `s` in a row: `exp(s)`, then divide by the row's sum. All weights come out positive and every row adds up to 1.
One trick every real implementation uses: **subtract the row's biggest score first**. The answer doesn't change (it cancels in the
division), but `exp` never sees a huge number, so it can't overflow. And `exp(-inf)` is exactly 0, which is how the mask works.

✏️ **YOUR TURN:** finish `softmax` below.

In [ ]:
def softmax(s):
    """softmax over the last axis: every row becomes positive weights that add up to 1"""
    s = s - s.max(axis=-1, keepdims=True)     # the subtract-the-max trick
    e = np.exp(s)
    return ___

weights = softmax(masked)                     # [1, 4, 4]
print((weights[0] * 100).round(1))
print("each row adds up to:", weights[0].sum(axis=-1))

In [ ]:
gpt2_w = model(torch.tensor(ids), output_attentions=True).attentions[0][0, HEAD].detach().numpy()   # layer 0, head 4: [4, 4]
print("largest difference from GPT-2's own weights:", np.abs(weights[0] - gpt2_w).max())
assert np.allclose(weights[0], gpt2_w, atol=1e-5) and np.allclose(weights.sum(-1), 1)
print("✅ your weights are GPT-2's weights for layer 0, head 4")

**Why the 8 matters, on our own rows.** Same masked table without dividing by 8: the weights get much more extreme.
(Row 0 is always 100%: `" The"` can only look at itself.)

In [ ]:
print("with / 8:   ", (weights[0, 3] * 100).round(1))
print("without / 8:", (softmax(np.where(keep, scores, -np.inf))[0, 3] * 100).round(1))

### 2.8 · Step 6: the weighted mix of values
Each token's new row = its weights × every value row, added up: `out[b, i] = Σ_j weights[b, i, j] · V[b, j]`. As one multiplication:
`weights [1, 4, 4] @ V [1, 4, 64] → out [1, 4, 64]`.

In [ ]:
out = ___                    # [1, 4, 64]
print("out", out.shape)
print("row 3 (' on') by hand, first 4 numbers:", sum(weights[0, 3, j] * V[0, j, :4] for j in range(4)).round(4))
print("row 3 from @,           first 4 numbers:", out[0, 3, :4].round(4))

GPT-2 joins its 12 heads' outputs side by side (12 × 64 = 768) and sends them into `c_proj`. We catch that input with a **hook**
(a function PyTorch calls while the model runs) and check our head 4 against its columns 256–319.

In [ ]:
caught = {}
hook = blk.attn.c_proj.register_forward_pre_hook(lambda m, inp: caught.__setitem__("heads", inp[0].detach().numpy()))
with torch.no_grad():
    model(torch.tensor(ids))
hook.remove()
gpt2_out = caught["heads"][:, :, HEAD * HS:(HEAD + 1) * HS]          # [1, 4, 64]: head 4's part
diff = np.abs(out - gpt2_out).max()
print("largest difference from GPT-2's head 4:", diff)
assert diff < 1e-5
print("✅ one GPT-2 attention head, done by hand, matches")

**All 12 heads.** The same six steps with each head's own columns. Every head makes its own weights from the same 4 rows:

In [ ]:
def head_by_hand(h, head):
    """h [B, T, 768] after ln_1 -> (weights [B, T, T], out [B, T, 64]) for one head of layer 0"""
    Q = h @ Wc[:, cols("q", head)] + bc[cols("q", head)]
    K = h @ Wc[:, cols("k", head)] + bc[cols("k", head)]
    V = h @ Wc[:, cols("v", head)] + bc[cols("v", head)]
    T = h.shape[1]
    keep = np.tril(np.ones((T, T), dtype=bool))
    w = softmax(np.where(keep, Q @ K.transpose(0, 2, 1) / math.sqrt(HS), -np.inf))
    return w, w @ V

fig, axes = plt.subplots(2, 6, figsize=(13, 4.6))
for hd, ax in enumerate(axes.flat):
    w, _ = head_by_hand(h, hd)
    ax.imshow(w[0], vmin=0, vmax=1, cmap="viridis")
    ax.set_title(f"head {hd}" + (" ★" if hd == HEAD else ""), fontsize=9)
    ax.set_xticks(range(4), pieces, fontsize=7, rotation=90); ax.set_yticks(range(4), pieces, fontsize=7)
plt.suptitle("layer 0, 'The cat sat on': rows = token looking, columns = token looked at"); plt.tight_layout(); plt.show()

all_out = np.concatenate([head_by_hand(h, hd)[1] for hd in range(12)], axis=-1)     # [1, 4, 768]
assert np.abs(all_out - caught["heads"]).max() < 1e-5
print("✅ all 12 heads by hand, joined side by side, match GPT-2:", all_out.shape)

Careful reading meaning into one layer-0 head on 4 words: what a head "does" can only be judged over a lot of text.
Chapter 3 runs all 12 heads properly and joins them.

### 2.9 · Pencil and paper
64 numbers per row is too many by hand, so here is a made-up head small enough for paper: 3 tokens, queries and keys of 4 numbers
(so `d_k` = 4 and we divide by √4 = 2), values of 2 numbers. Work out the score table and the weights of row 2 on paper first,
then run the cell.

In [ ]:
tQ = np.array([[1, 0, 1, 0], [0, 2, 0, 0], [1, 1, 1, 1]], float)
tK = np.array([[1, 1, 0, 0], [0, 2, 0, 2], [1, 0, 1, 0]], float)
tV = np.array([[1, 0], [0, 1], [2, 2]], float)
t_scores = tQ @ tK.T                                    # [3, 3]
t_w = softmax(np.where(np.tril(np.ones((3, 3), dtype=bool)), t_scores / 2, -np.inf))
print("scores:\n", t_scores, "\n/ 2 and masked, then softmax:\n", t_w.round(4), "\nout = weights @ V:\n", (t_w @ tV).round(4))

### 2.10 · A real training batch: `[8, 64] → [8, 64, 64]`
Training never runs one sentence. Here is the first `[8, 64]` batch of our TinyStories file (the first 512 IDs of `train.bin`, cut
into 8 rows as in chapter 1). Nothing in the six steps changes: `@` and broadcasting handle the extra batch rows, every row gets its
own `[64, 64]` table, and the **same** `[64, 64]` causal mask is used on all 8.

In [ ]:
batch_ids = np.asarray(train_data[:8 * 64]).reshape(8, 64).astype(np.int64)    # [8, 64]
hb = layer_norm(embed(batch_ids), gamma, beta)                                 # [8, 64, 768]: Step 0, your LayerNorm
wb, outb = head_by_hand(hb, HEAD)
print("ids", batch_ids.shape, "-> h", hb.shape, "-> weights", wb.shape, "-> out", outb.shape)
print(f"{wb.size:,} weights in one head's table for this batch (8 × 64 × 64)")

with torch.no_grad():
    gw = model(torch.tensor(batch_ids), output_attentions=True).attentions[0][:, HEAD].numpy()   # [8, 64, 64]
assert np.abs(wb - gw).max() < 1e-5
print("✅ the same head on a real [8, 64] batch matches GPT-2, all 8 rows")

**Packed rows hold more than one story.** A row that contains an end-of-text token (ID 50256) has the end of one story and the
start of the next. The causal mask only stops looking *forward*, so the new story's tokens can still look back at the old one:

In [ ]:
EOT = tok.eos_token_id
b = next(r for r in range(8) if (batch_ids[r] == EOT).any())
e = int(np.where(batch_ids[b] == EOT)[0][0])
print(f"row {b}: end-of-text at position {e}")
print("   ...", tok.decode(batch_ids[b, max(0, e - 8):e + 1].tolist()), "|", tok.decode(batch_ids[b, e + 1:e + 9].tolist()), "...")
back = wb[b, e + 1:, :e + 1].sum(axis=-1)          # weight each new-story token gives to the old story + its end-of-text
print(f"head 4: tokens of the new story give {back.mean():.1%} of their weight, on average, to positions 0..{e}")
plt.imshow(wb[b], cmap="viridis", vmin=0, vmax=0.5); plt.axvline(e, color="w", lw=0.6); plt.axhline(e, color="w", lw=0.6)
plt.title(f"row {b}, head 4: weights [64, 64]"); plt.xlabel("key position (looked at)"); plt.ylabel("query position (looking)"); plt.show()

That's standard in packed pretraining (GPT-2's training data was packed this way in nanoGPT, and many models train like this).
Some models add a **document mask** that also blocks looking across the end-of-text; Llama 3 reports it had limited impact in
ordinary pretraining. Our training will use the plain causal mask.

### 2.11 · Chapter 2 toolkit
Later chapters start here: `layer_norm()`, `softmax()`, `cols()` and `head_by_hand()` for any head of layer 0, without redoing the exercises.
(Run the chapter 1 toolkit first.)

In [ ]:
import math
model.set_attn_implementation("eager")
blk = model.transformer.h[0]
Wc = blk.attn.c_attn.weight.detach().numpy()     # [768, 2304]
bc = blk.attn.c_attn.bias.detach().numpy()       # [2304]
HS = 64

def layer_norm(x, gamma, beta, eps=1e-5):
    """LayerNorm over the last axis: each row -> mean 0, spread 1, then * gamma + beta"""
    mu = x.mean(axis=-1, keepdims=True)
    var = ((x - mu) ** 2).mean(axis=-1, keepdims=True)
    return (x - mu) / np.sqrt(var + eps) * gamma + beta

def softmax(s):
    s = s - s.max(axis=-1, keepdims=True)
    e = np.exp(s)
    return e / e.sum(axis=-1, keepdims=True)

def cols(kind, head):
    start = {"q": 0, "k": 768, "v": 1536}[kind] + head * HS
    return slice(start, start + HS)

def head_by_hand(h, head):
    """h [B, T, 768] after ln_1 -> (weights [B, T, T], out [B, T, 64]) for one head of layer 0"""
    Q = h @ Wc[:, cols("q", head)] + bc[cols("q", head)]
    K = h @ Wc[:, cols("k", head)] + bc[cols("k", head)]
    V = h @ Wc[:, cols("v", head)] + bc[cols("v", head)]
    T = h.shape[1]
    keep = np.tril(np.ones((T, T), dtype=bool))
    w = softmax(np.where(keep, Q @ K.transpose(0, 2, 1) / math.sqrt(HS), -np.inf))
    return w, w @ V

gamma, beta = blk.ln_1.weight.detach().numpy(), blk.ln_1.bias.detach().numpy()
h_cat = layer_norm(embed([[464, 3797, 3332, 319]]), gamma, beta)
print("head 4 weights of ' on':", (head_by_hand(h_cat, 4)[0][0, 3] * 100).round(1))

### 2.12 · TinyGPT, part 2: its LayerNorm and one attention head
Chapter 1 gave TinyGPT its two tables. Now it gets the pieces we just built: **Step 0's LayerNorm** (PyTorch has it built in:
`nn.LayerNorm(384)`, starting at γ = 1, β = 0 like OpenAI's code) and **one attention head** as a PyTorch module. With 6 heads
over 384 numbers, each head is 384 / 6 = 64 wide, the same size as a GPT-2 head.

| new | what it is |
|---|---|
| `nn.Linear(384, 64)` | a table `[64, 384]` plus a bias `[64]`; `self.query(x)` computes `x @ Wᵀ + b`. PyTorch stores it as `[out, in]`, the transpose of GPT-2's `[768, 64]` slices |
| `register_buffer` | keeps the mask inside the module (it moves to the GPU with it) without making it a weight training would change |
| `~self.causal[:T, :T]` | the top-left `T × T` corner of the stored mask, flipped (`~`): True where looking is **not** allowed |

The first cell repeats TinyGPT from 1.15 (filled in), so this section runs on its own.

In [ ]:
import torch.nn as nn
from dataclasses import dataclass

@dataclass
class TinyGPTConfig:
    vocab_size: int = 50257
    block_size: int = 256
    n_layer: int = 6
    n_head: int = 6
    n_embd: int = 384

class TinyGPT(nn.Module):
    """chapter 1: token table + position table. Chapters 3-4 put the heads inside it."""
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.wte = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.wpe = nn.Embedding(cfg.block_size, cfg.n_embd)
        nn.init.normal_(self.wte.weight, mean=0.0, std=0.02)
        nn.init.normal_(self.wpe.weight, mean=0.0, std=0.01)

    def forward(self, idx):
        B, T = idx.shape
        assert T <= self.cfg.block_size
        return self.wte(idx) + self.wpe(torch.arange(T, device=idx.device))

In [ ]:
class Head(nn.Module):
    """One causal self-attention head: [B, T, n_embd] -> [B, T, head_size]."""
    def __init__(self, cfg, n_in=None):
        super().__init__()
        n_in = n_in or cfg.n_embd
        hs = cfg.n_embd // cfg.n_head                 # 384 // 6 = 64
        self.query = nn.Linear(n_in, hs)              # Wq and bq
        self.key   = nn.Linear(n_in, hs)              # Wk and bk
        self.value = nn.Linear(n_in, hs)              # Wv and bv
        for lin in (self.query, self.key, self.value):   # start random, OpenAI's GPT-2 recipe: normal, std 0.02; biases 0
            nn.init.normal_(lin.weight, mean=0.0, std=0.02)
            nn.init.zeros_(lin.bias)
        # the causal mask: True on and below the diagonal, for the longest text TinyGPT reads
        self.register_buffer("causal", torch.tril(torch.ones(cfg.block_size, cfg.block_size, dtype=torch.bool)))

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.query(x), self.key(x), self.value(x)          # step 1: [B, T, hs] each
        att = q @ k.transpose(-2, -1) / math.sqrt(k.size(-1))        # steps 2-3: [B, T, T]
        att = att.masked_fill(~self.causal[:T, :T], ___)             # step 4
        att = torch.softmax(att, dim=-1)                             # step 5
        return ___                                                   # step 6: [B, T, hs]

✏️ **YOUR TURN:** fill the two blanks (Hint: the same −∞ and the same weighted mix as 2.6 and 2.8, in PyTorch). Then feed a real
TinyStories batch through TinyGPT's tables, its LayerNorm and one random head.

In [ ]:
torch.manual_seed(0)
cfg = TinyGPTConfig()
tiny, ln_1, head = TinyGPT(cfg), nn.LayerNorm(cfg.n_embd), Head(cfg)
print(f"LayerNorm: {sum(p.numel() for p in ln_1.parameters()):,} weights (γ and β, 384 each), starting at 1 and 0")
print(f"one head: {sum(p.numel() for p in head.parameters()):,} weights (3 × (384 × 64 + 64)), all random")
xb = torch.tensor(get_batch(train_data, 8, 64))       # a real [8, 64] batch
rows = tiny(xb)                                       # [8, 64, 384]
hrows = ln_1(rows)                                    # [8, 64, 384]: Step 0
hout = head(hrows)                                    # [8, 64, 64]:  Steps 1-6
print(list(xb.shape), "->", list(rows.shape), "-> LayerNorm ->", list(hrows.shape), "-> head ->", list(hout.shape))
assert hout.shape == (8, 64, 64) and sum(p.numel() for p in head.parameters()) == 73_920
print("✅ TinyGPT's rows go through its LayerNorm and one attention head: [8, 64, 384] -> [8, 64, 64]")

**Is our code right? The same one-time test as chapter 1.** A throwaway LayerNorm + head at GPT-2's size (768 in, 64 out, 1,024
positions) gets GPT-2's layer-0 `ln_1` and head-4 weights (the head's **transposed** into `nn.Linear`'s `[out, in]` layout), and runs on
the real batch from 2.10. Then both are deleted: TinyGPT keeps its own random ones.

In [ ]:
check = Head(TinyGPTConfig(block_size=1024, n_head=12, n_embd=768))
check_ln = nn.LayerNorm(768)
with torch.no_grad():
    check_ln.weight.copy_(blk.ln_1.weight); check_ln.bias.copy_(blk.ln_1.bias)
    for name, kind in (("query", "q"), ("key", "k"), ("value", "v")):
        lin = getattr(check, name)
        lin.weight.copy_(torch.tensor(Wc[:, cols(kind, 4)].T))     # [768, 64] -> [64, 768]
        lin.bias.copy_(torch.tensor(bc[cols(kind, 4)]))
    ours = check(check_ln(torch.tensor(embed(batch_ids)))).numpy()  # the 2.10 batch: embed -> LayerNorm -> head
caught = {}
hook = blk.attn.c_proj.register_forward_pre_hook(lambda m, inp: caught.__setitem__("heads", inp[0].detach().numpy()))
with torch.no_grad():
    model(torch.tensor(batch_ids))
hook.remove()
diff = np.abs(ours - caught["heads"][:, :, 256:320]).max()
print("largest difference from GPT-2's head 4 on the whole [8, 64] batch:", diff)
assert diff < 1e-5
del check, check_ln
print("✅ our LayerNorm + Head compute exactly what GPT-2 computes. Test copies deleted; TinyGPT keeps its random ones")

#### Chapter 2 solutions
```python
var        = ((x - mu) ** 2).mean(axis=-1, keepdims=True)     # layer_norm
Wv, bv     = Wc[:, cols("v", HEAD)], bc[cols("v", HEAD)]
Q          = h @ Wq + bq
one_score  = np.sum(Q[0, 3] * K[0, 2])
scores     = Q @ K.transpose(0, 2, 1)
masked     = np.where(keep, scaled, -np.inf)
return e / e.sum(axis=-1, keepdims=True)          # softmax
out        = weights @ V
# Head.forward
att = att.masked_fill(~self.causal[:T, :T], float("-inf"))
return att @ v
```
**Next chapter:** one full block: all heads side by side, the output projection, the MLP with its own LayerNorm (`ln_2`, the same
recipe as 2.1), and the skip connections. And TinyGPT gets its first complete block.